In [23]:
#-------------------------------------------------------------------------------
# Download datatsets
#-------------------------------------------------------------------------------

import kagglehub

# Disease dataset
disease_path = kagglehub.dataset_download("adiithape1/plant-disease-detection-dataset-master-version")
print("Path to  disease dataset:", disease_path)

# Crop dataset
crop_path = kagglehub.dataset_download("omrathod2003/140-most-popular-crops-image-dataset")
print("Path to  crop dataset:", crop_path)


Using Colab cache for faster access to the 'plant-disease-detection-dataset-master-version' dataset.
Path to  disease dataset: /kaggle/input/plant-disease-detection-dataset-master-version
Using Colab cache for faster access to the '140-most-popular-crops-image-dataset' dataset.
Path to  crop dataset: /kaggle/input/140-most-popular-crops-image-dataset


In [24]:
#-------------------------------------------------------------------------------
# Copying datasets into content/ from session's cache/
#-------------------------------------------------------------------------------
import os
import shutil

# Source directories
CROP_SRC = "/root/.cache/kagglehub/datasets/omrathod2003/140-most-popular-crops-image-dataset/versions/5/RGB_224x224/RGB_224x224"
DISEASE_SRC = "/root/.cache/kagglehub/datasets/adiithape1/plant-disease-detection-dataset-master-version/versions/2/MasterDataset"

# Target directories
CROP_DST = "/content/crop_dataset"
DISEASE_DST = "/content/disease_dataset"

def move_dataset(src, dst):
    if os.path.exists(dst):
        print(f"{dst} already exists. Skipping copy.")
    else:
        shutil.copytree(src, dst)
        print(f"Copied dataset to {dst}")

# Move datasets
move_dataset(CROP_SRC, CROP_DST)
move_dataset(DISEASE_SRC, DISEASE_DST)


Copied dataset to /content/crop_dataset
Copied dataset to /content/disease_dataset


In [25]:
CROP_WORKDIR = "/content/crop_dataset"
DISEASE_WORKDIR = "/content/disease_dataset"

#-------------------------------------------------------------------------------
# Deleting unwanted crops from 139 crops
#-------------------------------------------------------------------------------


all_crop_folders = os.listdir("/content/crop_dataset/train")

removed = []
kept = []

SELECTED_CROPS = {
    "Apples plant",
    "Cherry plant",
    "Cucumbers and gherkins plant",
    "Grapes plant",
    "Groundnuts (Peanuts) plant",
    "Guavas plant",
    "Lemons and limes plant",
    "Maize (Corn) plant",
    "Peaches and nectarines plant",
    "Potatoes plant",
    "Pumpkins, squash and gourds plant",
    "Rice (Paddy) plant",
    "Strawberries plant",
    "Tomatoes plant",
    "Wheat plant"
}

for crop in all_crop_folders:
    crop_path_train = os.path.join("/content/crop_dataset/train", crop)
    crop_path_test = os.path.join("/content/crop_dataset/test", crop)
    crop_path_val = os.path.join("/content/crop_dataset/val", crop)

    if not os.path.isdir(crop_path_test):
        continue
    if not os.path.isdir(crop_path_train):
        continue
    if not os.path.isdir(crop_path_val):
        continue

    if crop in SELECTED_CROPS:
        kept.append(crop)
    else:
        shutil.rmtree(crop_path_train)
        shutil.rmtree(crop_path_test)
        shutil.rmtree(crop_path_val)
        removed.append(crop)

print(f"Kept {len(kept)} crops:")
print(sorted(kept))

print(f"\nRemoved {len(removed)} crops")


Kept 15 crops:
['Apples plant', 'Cherry plant', 'Cucumbers and gherkins plant', 'Grapes plant', 'Groundnuts (Peanuts) plant', 'Guavas plant', 'Lemons and limes plant', 'Maize (Corn) plant', 'Peaches and nectarines plant', 'Potatoes plant', 'Pumpkins, squash and gourds plant', 'Rice (Paddy) plant', 'Strawberries plant', 'Tomatoes plant', 'Wheat plant']

Removed 124 crops


In [26]:
#-------------------------------------------------------------------------------
# Extracting the crop and disease classes.
#-------------------------------------------------------------------------------


raw_diseases = list(os.listdir("/content/disease_dataset/train"))
parsed = []
EXCLUDE = ["bean","cotton","bell","bell_pepper","sugarcane"]
for label in raw_diseases:
  crop,disease = label.split("_",1)

  if crop in EXCLUDE or disease in EXCLUDE:
    continue

  if crop == "healthy" or crop == "diseased":
    crop,disease = disease,crop
    if disease == "diseased":
      disease = "unhealthy"

  parsed.append((crop,disease))

CROP_CLASSES = sorted({crop for crop,_ in parsed})
DISEASE_CLASSES = sorted({disease for _,disease in parsed})

print(f"Crops: {CROP_CLASSES}")
print(f"Diseases: {DISEASE_CLASSES}")


Crops: ['apple', 'cherry', 'corn', 'cucumber', 'grape', 'groundnut', 'guava', 'lemon', 'peach', 'potato', 'pumpkin', 'rice', 'strawberry', 'tomato', 'wheat']
Diseases: ['anthracnose', 'aphid', 'apple_scab', 'bacterial_blight', 'bacterial_leaf_spot', 'bacterial_spot', 'black_rot', 'black_rust', 'blast', 'brown_rust', 'cedar_apple_rust', 'cercospora_leaf_spot', 'citrus_canker', 'common_root_rot', 'common_rust', 'curl_virus', 'deficiency', 'downy_mildew', 'dry_leaf', 'early_blight', 'early_leaf_spot', 'esca_black_measles', 'fruit_fly', 'fusarium_head_blight', 'gray_leaf_spot', 'healthy', 'late_blight', 'late_leaf_spot', 'leaf_blight', 'leaf_scorch', 'mildew', 'mite', 'mosaic_disease', 'northern_leaf_blight', 'nutrition_deficiency', 'powdery_mildew', 'rust', 'septoria', 'septoria_leaf_spot', 'smut', 'sooty_mould', 'spider_mites', 'stem_fly', 'tan_spot', 'unhealthy', 'yellow_leaf_curl_virus', 'yellow_rust']


In [27]:
#-------------------------------------------------------------------------------
# Renaming directory names in crop dataset.
#-------------------------------------------------------------------------------

import os

CROP_WORKDIR = "/content/crop_dataset"
SPLITS = ["train", "test", "val"]

RENAME_MAP = {
    "Apples plant": "apple",
    "Cherry plant": "cherry",
    "Cucumbers and gherkins plant": "cucumber",
    "Grapes plant": "grape",
    "Groundnuts (Peanuts) plant": "groundnut",
    "Guavas plant": "guava",
    "Lemons and limes plant": "lemon",
    "Maize (Corn) plant": "corn",
    "Peaches and nectarines plant": "peach",
    "Potatoes plant": "potato",
    "Pumpkins, squash and gourds plant": "pumpkin",
    "Rice (Paddy) plant": "rice",
    "Strawberries plant": "strawberry",
    "Tomatoes plant": "tomato",
    "Wheat plant": "wheat"
}


for split in SPLITS:
    split_dir = os.path.join(CROP_WORKDIR, split)
    for old_name, new_name in RENAME_MAP.items():
        old_path = os.path.join(split_dir, old_name)
        new_path = os.path.join(split_dir, new_name)

        if os.path.exists(old_path):
            if not os.path.exists(new_path):
                os.rename(old_path, new_path)
                print(f"[{split}] Renamed: {old_name} → {new_name}")
            else:
                print(f"[{split}] {new_name} already exists, skipping")


[train] Renamed: Apples plant → apple
[train] Renamed: Cherry plant → cherry
[train] Renamed: Cucumbers and gherkins plant → cucumber
[train] Renamed: Grapes plant → grape
[train] Renamed: Groundnuts (Peanuts) plant → groundnut
[train] Renamed: Guavas plant → guava
[train] Renamed: Lemons and limes plant → lemon
[train] Renamed: Maize (Corn) plant → corn
[train] Renamed: Peaches and nectarines plant → peach
[train] Renamed: Potatoes plant → potato
[train] Renamed: Pumpkins, squash and gourds plant → pumpkin
[train] Renamed: Rice (Paddy) plant → rice
[train] Renamed: Strawberries plant → strawberry
[train] Renamed: Tomatoes plant → tomato
[train] Renamed: Wheat plant → wheat
[test] Renamed: Apples plant → apple
[test] Renamed: Cherry plant → cherry
[test] Renamed: Cucumbers and gherkins plant → cucumber
[test] Renamed: Grapes plant → grape
[test] Renamed: Groundnuts (Peanuts) plant → groundnut
[test] Renamed: Guavas plant → guava
[test] Renamed: Lemons and limes plant → lemon
[test] Ren

In [28]:
#-------------------------------------------------------------------------------
# Indexing classes.
#-------------------------------------------------------------------------------


crop_to_index = {crop: i for i, crop in enumerate(CROP_CLASSES)}
index_to_crop = {i: crop for crop, i in crop_to_index.items()}

disease_to_index = {disease: i for i, disease in enumerate(DISEASE_CLASSES)}
index_to_disease = {i: disease for disease, i in disease_to_index.items()}

print(f"Crop to index: {crop_to_index}")
print(f"Index to crop: {index_to_crop}")
print(f"Disease to index: {disease_to_index}")
print(f"Index to disease: {index_to_disease}")


Crop to index: {'apple': 0, 'cherry': 1, 'corn': 2, 'cucumber': 3, 'grape': 4, 'groundnut': 5, 'guava': 6, 'lemon': 7, 'peach': 8, 'potato': 9, 'pumpkin': 10, 'rice': 11, 'strawberry': 12, 'tomato': 13, 'wheat': 14}
Index to crop: {0: 'apple', 1: 'cherry', 2: 'corn', 3: 'cucumber', 4: 'grape', 5: 'groundnut', 6: 'guava', 7: 'lemon', 8: 'peach', 9: 'potato', 10: 'pumpkin', 11: 'rice', 12: 'strawberry', 13: 'tomato', 14: 'wheat'}
Disease to index: {'anthracnose': 0, 'aphid': 1, 'apple_scab': 2, 'bacterial_blight': 3, 'bacterial_leaf_spot': 4, 'bacterial_spot': 5, 'black_rot': 6, 'black_rust': 7, 'blast': 8, 'brown_rust': 9, 'cedar_apple_rust': 10, 'cercospora_leaf_spot': 11, 'citrus_canker': 12, 'common_root_rot': 13, 'common_rust': 14, 'curl_virus': 15, 'deficiency': 16, 'downy_mildew': 17, 'dry_leaf': 18, 'early_blight': 19, 'early_leaf_spot': 20, 'esca_black_measles': 21, 'fruit_fly': 22, 'fusarium_head_blight': 23, 'gray_leaf_spot': 24, 'healthy': 25, 'late_blight': 26, 'late_leaf_sp

In [29]:
#-------------------------------------------------------------------------------
# Crop -> Disease validity mapping.
#-------------------------------------------------------------------------------

from collections import defaultdict

crop_to_diseases = defaultdict(set)

for crop, disease in parsed:
    crop_to_diseases[crop].add(disease)

# convert to sorted lists
crop_to_diseases = {
    crop: sorted(list(diseases))
    for crop, diseases in crop_to_diseases.items()
}

# sanity check
for crop in sorted(crop_to_diseases):
    print(crop, "→", crop_to_diseases[crop])


apple → ['apple_scab', 'black_rot', 'cedar_apple_rust', 'healthy']
cherry → ['healthy', 'powdery_mildew']
corn → ['cercospora_leaf_spot', 'common_rust', 'gray_leaf_spot', 'healthy', 'northern_leaf_blight']
cucumber → ['healthy', 'unhealthy']
grape → ['black_rot', 'esca_black_measles', 'healthy', 'leaf_blight']
groundnut → ['early_leaf_spot', 'healthy', 'late_leaf_spot', 'nutrition_deficiency', 'rust']
guava → ['anthracnose', 'fruit_fly', 'healthy']
lemon → ['anthracnose', 'bacterial_blight', 'citrus_canker', 'curl_virus', 'deficiency', 'dry_leaf', 'healthy', 'sooty_mould', 'spider_mites']
peach → ['bacterial_spot', 'healthy']
potato → ['early_blight', 'healthy', 'late_blight']
pumpkin → ['bacterial_leaf_spot', 'downy_mildew', 'healthy', 'mosaic_disease', 'powdery_mildew']
rice → ['healthy', 'unhealthy']
strawberry → ['healthy', 'leaf_scorch']
tomato → ['bacterial_spot', 'early_blight', 'healthy', 'late_blight', 'septoria_leaf_spot', 'yellow_leaf_curl_virus']
wheat → ['aphid', 'black_ru

In [30]:
#-------------------------------------------------------------------------------
# Crop -> Disease masking.
#-------------------------------------------------------------------------------

import numpy as np

num_crops = len(CROP_CLASSES)
num_diseases = len(DISEASE_CLASSES)

# Initialize mask with zeros
crop_disease_mask = np.zeros((num_crops, num_diseases), dtype=np.float32)

for crop, diseases in crop_to_diseases.items():
    crop_idx = crop_to_index[crop]

    for disease in diseases:
        disease_idx = disease_to_index[disease]
        crop_disease_mask[crop_idx, disease_idx] = 1.0

print("Mask shape:", crop_disease_mask.shape)

# Validating the mask
def show_valid_diseases(crop_name):
    crop_idx = crop_to_index[crop_name]
    valid = [
        DISEASE_CLASSES[i]
        for i in range(num_diseases)
        if crop_disease_mask[crop_idx, i] == 1
    ]
    print(crop_name, "→", valid)

show_valid_diseases("rice")
show_valid_diseases("cucumber")
show_valid_diseases("tomato")


Mask shape: (15, 47)
rice → ['healthy', 'unhealthy']
cucumber → ['healthy', 'unhealthy']
tomato → ['bacterial_spot', 'early_blight', 'healthy', 'late_blight', 'septoria_leaf_spot', 'yellow_leaf_curl_virus']


In [31]:
#-------------------------------------------------------------------------------
# Loss function definition for Disease <-> Crop.
#-------------------------------------------------------------------------------

import tensorflow as tf

crop_disease_mask_tf = tf.constant(crop_disease_mask)

def masked_disease_loss(y_true, y_pred, crop_true):
    """
    y_true: one-hot disease labels (batch, num_diseases)
    y_pred: predicted logits (batch, num_diseases)
    crop_true: one-hot crop labels (batch, num_crops)
    """

    # Get crop indices
    crop_indices = tf.argmax(crop_true, axis=1)

    # Gather mask for each sample
    masks = tf.gather(crop_disease_mask_tf, crop_indices)

    # Apply mask to predictions
    masked_logits = y_pred * masks + (1.0 - masks) * (-1e9)

    # Standard categorical cross-entropy
    loss = tf.keras.losses.categorical_crossentropy(
        y_true, masked_logits, from_logits=True
    )

    return loss

In [32]:
#-------------------------------------------------------------------------------
# Build the Model.
#-------------------------------------------------------------------------------

import tensorflow as tf
from tensorflow.keras import layers, models

IMG_SIZE = (224, 224)
NUM_CROPS = len(CROP_CLASSES)
NUM_DISEASES = len(DISEASE_CLASSES)

base_model = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=IMG_SIZE + (3,)
)

base_model.trainable = False  # freeze backbone for now

# Shared feature extractor
inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)

# Crop head
crop_logits = layers.Dense(
    NUM_CROPS,
    activation=None,
    name="crop_logits"
)(x)

crop_output = layers.Activation(
    "softmax",
    name="crop_output"
)(crop_logits)

# Disease head
disease_logits = layers.Dense(
    NUM_DISEASES,
    activation=None,
    name="disease_logits"
)(x)

# Assembling the model
model = models.Model(
    inputs=inputs,
    outputs=[crop_output, disease_logits]
)

model.summary()

# Wrapping masked_disease_loss with crop_labels
def disease_loss_wrapper(crop_true):
    def loss(y_true, y_pred):
        return masked_disease_loss(y_true, y_pred, crop_true)
    return loss

# Compiling
crop_loss_fn = tf.keras.losses.CategoricalCrossentropy()
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)

model.compile(
    optimizer=optimizer,
    loss={
        "crop_output": crop_loss_fn,
        "disease_logits": None  # handled manually
    },
    metrics={
        "crop_output": "accuracy"
    }
)


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ efficientnetb0      │ (None, 7, 7,      │  4,049,571 │ input_layer_3[0]… │
│ (Functional)        │ 1280)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 1280)      │          0 │ efficientnetb0[0… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 1280)      │      5,120 │ global_average_p… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ crop_logits (Dense) │ (None, 15)        │     19,215 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ crop_output         │ (None, 15)        │          0 │ crop_logits[0][0] │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ disease_logits      │ (None, 47)        │     60,207 │ batch_normalizat… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 4,134,113 (15.77 MB)

 Trainable params: 81,982 (320.24 KB)

 Non-trainable params: 4,052,131 (15.46 MB)

In [33]:
#-------------------------------------------------------------------------------
# Custom training model
#-------------------------------------------------------------------------------

class MultiTaskTrainer(tf.keras.Model):

    def __init__(self, model, **kwargs):
        super().__init__(**kwargs)
        self.model = model

        # Loss for crop classification
        self.crop_loss_fn = tf.keras.losses.CategoricalCrossentropy()

        # Metrics
        self.crop_accuracy = tf.keras.metrics.CategoricalAccuracy(name="crop_accuracy")
        self.total_loss_tracker = tf.keras.metrics.Mean(name="loss")

    def train_step(self, data):
      images, labels = data

      # task is a Tensor → convert to Python string (eager mode)
      task = labels["task"][0].numpy().decode()

      with tf.GradientTape() as tape:
          crop_pred, disease_pred = self.model(images, training=True)

          if task == "crop":
              crop_true = labels["crop_output"]
              loss = self.crop_loss_fn(crop_true, crop_pred)

          else:  # task == "disease"
              crop_true = labels["crop_output"]
              disease_true = labels["disease_logits"]

              loss = masked_disease_loss(
                  disease_true,
                  disease_pred,
                  crop_true
              )

          loss = tf.reduce_mean(loss)

      grads = tape.gradient(loss, self.model.trainable_variables)
      self.optimizer.apply_gradients(
          zip(grads, self.model.trainable_variables)
      )

      if task == "crop":
          self.crop_accuracy.update_state(crop_true, crop_pred)

      self.total_loss_tracker.update_state(loss)

      return {
          "loss": self.total_loss_tracker.result(),
          "crop_accuracy": self.crop_accuracy.result()
      }

    def test_step(self, data):
      images, labels = data
      task = labels["task"][0].numpy().decode()

      crop_pred, disease_pred = self.model(images, training=False)

      if task == "crop":
          crop_true = labels["crop_output"]
          loss = self.crop_loss_fn(crop_true, crop_pred)
          self.crop_accuracy.update_state(crop_true, crop_pred)

      else:
          crop_true = labels["crop_output"]
          disease_true = labels["disease_logits"]
          loss = masked_disease_loss(
              disease_true,
              disease_pred,
              crop_true
          )

      loss = tf.reduce_mean(loss)
      self.total_loss_tracker.update_state(loss)

      return {
          "loss": self.total_loss_tracker.result(),
          "crop_accuracy": self.crop_accuracy.result()
      }

    @property
    def metrics(self):
        return [self.total_loss_tracker, self.crop_accuracy]


trainer = MultiTaskTrainer(model)
trainer.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    run_eagerly=True
)





In [34]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 8
AUTOTUNE = tf.data.AUTOTUNE

In [35]:
def load_and_preprocess_image(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.keras.applications.efficientnet.preprocess_input(img)
    return img


In [36]:
def build_crop_samples(root_dir):
    samples = []
    for crop in os.listdir(root_dir):
        crop_dir = os.path.join(root_dir, crop)
        if not os.path.isdir(crop_dir):
            continue

        for fname in os.listdir(crop_dir):
            if fname.lower().endswith(".jpg"):
                samples.append(
                    (os.path.join(crop_dir, fname), crop)
                )
    return samples


CROP_TRAIN_DIR = "/content/crop_dataset/train"
CROP_VAL_DIR   = "/content/crop_dataset/val"

crop_train_samples = build_crop_samples(CROP_TRAIN_DIR)
crop_val_samples   = build_crop_samples(CROP_VAL_DIR)

print("Crop train samples:", len(crop_train_samples))
print("Crop val samples:", len(crop_val_samples))

print("Example crop sample:", crop_train_samples[0])


Crop train samples: 2971
Crop val samples: 653
Example crop sample: ('/content/crop_dataset/train/corn/14753.jpg', 'corn')


In [37]:
def build_disease_samples(root_dir):
    samples = []

    VALID_CROPS = set(CROP_CLASSES)   # only crops you care about

    for folder in os.listdir(root_dir):
        folder_path = os.path.join(root_dir, folder)
        if not os.path.isdir(folder_path):
            continue

        first, second = folder.split("_", 1)

        # Handle healthy_* and diseased_*
        if first in {"healthy", "diseased"}:
            crop = second
            disease = first
        else:
            crop = first
            disease = second

        # Normalize diseased → unhealthy
        if disease == "diseased":
            disease = "unhealthy"

        # 🚫 EXCLUDE crops not in crop dataset
        if crop not in VALID_CROPS:
            continue

        for fname in os.listdir(folder_path):
            if fname.lower().endswith(".jpg"):
                samples.append(
                    (os.path.join(folder_path, fname), crop, disease)
                )

    return samples




DISEASE_TRAIN_DIR = "/content/disease_dataset/train"
DISEASE_VAL_DIR   = "/content/disease_dataset/val"

disease_train_samples = build_disease_samples(DISEASE_TRAIN_DIR)
disease_val_samples   = build_disease_samples(DISEASE_VAL_DIR)

print("Disease train samples:", len(disease_train_samples))
print("Disease val samples:", len(disease_val_samples))

print("Example disease sample:", disease_train_samples[0])


Disease train samples: 78662
Disease val samples: 9811
Example disease sample: ('/content/disease_dataset/train/diseased_rice/285a28ec-5fbc-58cb-9828-dcc6eb08eb48.jpg', 'rice', 'unhealthy')


In [38]:
def crop_generator(samples):
    for img_path, crop in samples:
        image = load_and_preprocess_image(img_path)

        crop_idx = crop_to_index[crop]

        yield image, {
            "task": tf.constant("crop"),
            "crop_output": tf.one_hot(crop_idx, NUM_CROPS),
            "disease_logits": tf.zeros(NUM_DISEASES, dtype=tf.float32),
        }


crop_train_ds = tf.data.Dataset.from_generator(
    lambda: crop_generator(crop_train_samples),
    output_signature=(
        tf.TensorSpec(shape=(224, 224, 3), dtype=tf.float32),
        {
            "task": tf.TensorSpec(shape=(), dtype=tf.string),
            "crop_output": tf.TensorSpec(shape=(NUM_CROPS,), dtype=tf.float32),
            "disease_logits": tf.TensorSpec(shape=(NUM_DISEASES,), dtype=tf.float32),
        }
    )
).shuffle(256).batch(BATCH_SIZE).prefetch(AUTOTUNE)


crop_train_ds = (
    crop_train_ds
    .shuffle(256)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)


crop_val_ds = tf.data.Dataset.from_generator(
    lambda: crop_generator(crop_val_samples),
    output_signature=(
        tf.TensorSpec(shape=(224, 224, 3), dtype=tf.float32),
        {
            "task": tf.TensorSpec(shape=(), dtype=tf.string),
            "crop_output": tf.TensorSpec(shape=(NUM_CROPS,), dtype=tf.float32),
            "disease_logits": tf.TensorSpec(shape=(NUM_DISEASES,), dtype=tf.float32),
        }
    )
).batch(BATCH_SIZE).prefetch(AUTOTUNE)


crop_val_ds = (
    crop_val_ds
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)


batch = next(iter(crop_train_ds))
images, labels = batch

print("Task:", labels["task"][0].numpy())
print("Images shape:", images.shape)
print("Crop label shape:", labels["crop_output"].shape)
print("One-hot sum:", tf.reduce_sum(labels["crop_output"][0]).numpy())


Task: [b'crop' b'crop' b'crop' b'crop' b'crop' b'crop' b'crop' b'crop' b'crop'
 b'crop' b'crop' b'crop' b'crop' b'crop' b'crop' b'crop' b'crop' b'crop'
 b'crop' b'crop' b'crop' b'crop' b'crop' b'crop' b'crop' b'crop' b'crop'
 b'crop' b'crop' b'crop' b'crop' b'crop']
Images shape: (32, 32, 224, 224, 3)
Crop label shape: (32, 32, 15)
One-hot sum: 32.0


In [39]:
def disease_generator(samples):
    for img_path, crop, disease in samples:
        image = load_and_preprocess_image(img_path)

        crop_idx = crop_to_index[crop]
        disease_idx = disease_to_index[disease]

        yield image, {
            "task": tf.constant("disease"),
            "crop_output": tf.one_hot(crop_idx, len(CROP_CLASSES)),
            "disease_logits": tf.one_hot(disease_idx, len(DISEASE_CLASSES)),
        }

disease_train_ds = tf.data.Dataset.from_generator(
    lambda: disease_generator(disease_train_samples),
    output_signature=(
        tf.TensorSpec(shape=(224, 224, 3), dtype=tf.float32),
        {
            "task": tf.TensorSpec(shape=(), dtype=tf.string),
            "crop_output": tf.TensorSpec(
                shape=(len(CROP_CLASSES),), dtype=tf.float32
            ),
            "disease_logits": tf.TensorSpec(
                shape=(len(DISEASE_CLASSES),), dtype=tf.float32
            ),
        }
    )
)

disease_train_ds = (
    disease_train_ds
    .shuffle(256)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)


disease_val_ds = tf.data.Dataset.from_generator(
    lambda: disease_generator(disease_val_samples),
    output_signature=(
        tf.TensorSpec(shape=(224, 224, 3), dtype=tf.float32),
        {
            "task": tf.TensorSpec(shape=(), dtype=tf.string),
            "crop_output": tf.TensorSpec(
                shape=(len(CROP_CLASSES),), dtype=tf.float32
            ),
            "disease_logits": tf.TensorSpec(
                shape=(len(DISEASE_CLASSES),), dtype=tf.float32
            ),
        }
    )
)

disease_val_ds = (
    disease_val_ds
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)


batch = next(iter(disease_train_ds))
images, labels = batch

print("Task:", labels["task"][0].numpy())
print("Images shape:", images.shape)
print("Crop label shape:", labels["crop_output"].shape)
print("Disease label shape:", labels["disease_logits"].shape)
print("Crop one-hot sum:", tf.reduce_sum(labels["crop_output"][0]).numpy())
print("Disease one-hot sum:", tf.reduce_sum(labels["disease_logits"][0]).numpy())


Task: b'disease'
Images shape: (32, 224, 224, 3)
Crop label shape: (32, 15)
Disease label shape: (32, 47)
Crop one-hot sum: 1.0
Disease one-hot sum: 1.0


In [40]:
train_ds = tf.data.Dataset.sample_from_datasets(
    [crop_train_ds, disease_train_ds],
    weights=[0.5, 0.5],   # can tune later (e.g., 0.6, 0.4)
    seed=42
)

val_ds = tf.data.Dataset.sample_from_datasets(
    [crop_val_ds, disease_val_ds],
    weights=[0.5, 0.5],
    seed=42
)


In [41]:
model.build((None, 224, 224, 3))
_ = model(tf.zeros((1, 224, 224, 3)), training=False)
model.summary()

history = trainer.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20
)


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ efficientnetb0      │ (None, 7, 7,      │  4,049,571 │ input_layer_3[0]… │
│ (Functional)        │ 1280)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 1280)      │          0 │ efficientnetb0[0… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 1280)      │      5,120 │ global_average_p… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ crop_logits (Dense) │ (None, 15)        │     19,215 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ crop_output         │ (None, 15)        │          0 │ crop_logits[0][0] │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ disease_logits      │ (None, 47)        │     60,207 │ batch_normalizat… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 4,134,113 (15.77 MB)

 Trainable params: 81,982 (320.24 KB)

 Non-trainable params: 4,052,131 (15.46 MB)

Epoch 1/20
      1/Unknown 13s 13s/step - crop_accuracy: 0.0000e+00 - loss: 1.5561

/usr/local/lib/python3.12/dist-packages/keras/src/optimizers/base_optimizer.py:855: UserWarning: Gradients do not exist for variables ['crop_logits/kernel', 'crop_logits/bias'] when minimizing the loss. If using `model.compile()`, did you forget to provide a `loss` argument?
  warnings.warn(


      4/Unknown 35s 8s/step - crop_accuracy: 0.0000e+00 - loss: 1.3818

AttributeError: 'numpy.ndarray' object has no attribute 'decode'